# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vl9v2zYQf9enIJSHyq0gy7LieB40IP/WBUiWokuAAUEgMBJtq5FEjqTcdEM/1973yXZHUrHcJun6MgG2KPL4u/vdHY/n+/4J06zQlNSUvH13TT7h4P3hBQxYXa0Y/BPB5LKqSUMVobXm5I+OkYLescjzjnlLupaSw0kck5KReUzeHsHeyewABzCDWIqRTtEB1MXh72cXlwtP85IrUsOv4SXDN4gCfElb0CBXFJdZ2xtnrNywP0OzRTDFlQcbhGRGuHWiqDIoaLFmaICgq6oF09+Qcck2Y7VuRpZlQ+uaAhT1NHvQnaSGM9lUJeNE0RrAjEIJeuquqICqcwGNPN/3vaoRXGoCRriRWne6qr3+Uyj7+f7y8opkIBexdlNJ3kYrpgP/8N30JMc1PyT+uOCtZq0eUzEt/ZF3eX2Vn5y9f24bLH+1Kwebq5Iq2H10fvrryemzu93yDsId8C2Z7N8A4nklW5J8JbpgtPAIPFp+sgN8HEnNZbHeTi7tRFR0JY0qldMNrWoKoD1G/wgwbiAKxuUQnqpguZAc0kRXTAXxaGePZBCmloiopQ0L4a25pnXesIbLT2RMktevp7HZwR4KJjQ5NS+I3Fa3oEp5AzD/+N01OCKOYkdY0iYvK3UfNFWbLyVj+eoumzvrIezHVAoGBwZSS0t6Rz9wTDuTc9sEUxVZU0gnJSgkzo/43fKQAG7BI8ydr9yJmsAlNoUiNCCHM7Nigd+j+qPICA15Op+b+Z8yMjB519slQD8C2Wxxxvs7cpAuDb1nZSVVUIbgxkrpnN9nV7JjT8YChFDhf3V6aVNSUL2OPvCqDTD/IQ17Y6yOb1ux1b4NXGkqWZ9nNkU2EE1QaZPY7jQT9mhGcDB095hBwcgm1I6HC9EpazSM8oJ3LaggXJLELH/k8j6EqDW5C+Bj+sABctExRkB0DuKFdceerZe2WI7JL/DhDc9FWQEPWyoz39ZKSFFeaFCR7U8SUKiZUNl+HJJi3bX3KktjfMKdENm8ogVT2WTfLBOodCD6wywED+Ixz5I4nYdQ0+qct+gl4+MnYPhyWXNaZj/TWqFfsUjmEpHjOeAq2ogavmZpaFzCpMqm1uFwjWxdMJ0vBg5I0QHPMz88vxrwTtHQr3gn3+Cd/D+8EaznnTzBO4kd7/OUJOmLrC9OT84GtKfztKedbmnPXmQ935K2PJ8jbVl9B+uDZEB6mjxBWrHFF2B75Colk/0XSR8d/nZ2PGCd7M961tMt6/lLpNNnSE/iJP0u0iYTnuM8mW05T9wR3yPYBjVdsaZ9u7PqqDRNDBcV9AxwVwybHLgv4NaBqgg9R8sdxobXGyax5WBMohTI2gsD+hLscphsKriqTRPi6qVBPrI3toMBAcStQREJ3JLpvlDjG7QvBJ1gWCMADd59dzWKDIK48bGEmfYpNwYr/9bWNUzkWdxLQdeF7so/smq11jtCadyXPjcBx3BwGdz4zoNmT0MfguFUSFzhEFEnSqpZgEHblnO8jiWU4TLAr5BMRiHZme5ndyOMdTzDPxs/qNEy2y3eW4R+BmF27hvxeNWoQlZ3LBDutnHrwdLHPvUvcfMKjH51+5kEOHZm4/fboxH552+TJrgyXMD53dt46aMMGo0SGEAQcS00rtghrsG8PTlm3g7d/JeQa6qge0Exc25e3S7Cz9hsQ1aBdN8K4zqMEUM8PAUz6H1QtvcpbIBL/F8appwq", "engine.py": "eNqtW+ty3LaS/j9PgWXKdch4RI9kn+Qc7WGqlFh2VCvbWsdJdqNVsTAkOAOLtxDkSCOXn2v/75NtdwMkQQ5nbCdxJRqSABqNRvfXF4KO47wq6qI6ZQn8uMpjR9+xjKcpZ0+fM/fHJt82PH/6/OiEPWEn2Ybaa3FfNxXHy+9Tkcei8mezn1XBYsEqETWVKtTpjLGv2bsiLhRL4f+siAX+/t6ImOcs4tWKY5vIGcz18upnxvFiIx6YW4oqkSl7dfZfF6/esCdAibGzy3dvvDnjSrK8YGu+ZUJBP44U6kqwJue0BLZFMkquGgkNwic2LmGiUgBXTAlWAos4e27mfnv2irkRj9YC+S/5SuZAFS4vZd7ceywVW1hjQVzgCngVreVGs17yiqewrlNmiFbAiFkrq3kVAyti1cBwpTk5T1ld8SV/D80iKwsYzjYcST2JxeaJWmfMjaWKCnwEnHlmAVpExH1Ri5VgUZHjVBHPY5Djv7cbwdwfoN9jGgp9KpCMyIn1llUaaYuIrUQOgoxkkfe70cutEqpJa9ormH7V4KKoI0pci02zymA1sDFqjbJL293oxfa+UbVQKKlS1LLCLplUmekoFJO5qnkO/4nCnzmOM5MooBoZBp2qgFc/aUDxoCtsUJS07auovcLZU7lsb2XRXr1XRd5eF6q9Uuumlml31yxBuJFQXXu9rgSPZb7qHshMtNcPsgQlFbP2Pm+ycouM5eUsqYqMXV1cMtN2kfGVmOufN6Wa6Q4+6xeYyNVs9uP523MWAIN+yeu1H8sq55lw23u+VPjrhiFOHIaeN4OdCi/f/PAfMKpj1r8solvXm/1ycf7rT9DgOjBbXjtz5ix5dIu/qUjovpKrde14s9ksSrlS7DwH5RentGexSFgYylzWYegqkSZz1L0keF3ksJS0WAVlJfPa073xH3byS5gR+zFQBr0uPxa1iGrgaNARKEBX+Dt8XIlsiQ04zbBFrXkp9reE2WayseTA5WSLVt6Affg4ZoHHW3j+gqdqNGSprSwsiyKFHlHivyOxX8H9+T1AH0Cpm/H78K6obkWlAs3CtWPunZteCrCtGb8VsMvKNZJ68/O78PnF2zkT91LVYXEbvKsaAfuD/b9iR3/in0bcbmcBBNOCx+GdQBVQZoPBNAsVuE4t8gjM7YkF/6guE4+zjeNZKgBWeykEqzvY78ASoZUQGHEI7t7zKu3gH9DDR4NvyZB5rJsVaOMq4ZEI182yNRYFkBfS1oUgun4ImISCLbm+6R7V1bbnjPqATlYAMyMarufTwoed8Z9MWEVtoYxx3J5uHW2xoV4wZCMV4Omevj2/j4FhVVdu4pNJo3l7RCrR023ouerXJO4jUdbsnH4QsdEBDmdprct1XhfkMZq4ACcmCHE7P/fji1PYUuENhlYC4DVni1n3FHWlil1ibDhNDsJeDJ7cyXrNilLk1B3hZQlTLJskEYAUq2DhIbfJrkzu1rBIhqo+La8lTKXN0v3mGfvXv9jJwpvsCPuVFzVb7he7WWE+2SHH/QALd5deLwHEDsR9H/9YIEarnQaA429opeJ+yEdd1BxRQzWZK+7B+Eu3iudaE7xddHQT50pbjIktTLgExnLKPmhiEJN9/fXTxal/nHxkL7/HZmd3aYnzwVoBO2L1qb9IPiqnn9TIhajOOpRAiDDQcAeeGSCWAMncEbJqhOoX2nrKAgy/lyHe+eh9AD6VHzUxh+XXWZP6EGoWd2GdPD0BySCtA4Py/Eu7LwGx1hmvbse9NcJsn8YQ/BifYzj/HiisqgJitrciKzaimh5DPgcu2mEdJj6X714Aj694DciXr65kKVJwqp9WKFBeywNKRR5rwrRbD7nD6AQt7TP30yKY+IGiYQAJO9p3AYUyPmfHFJ55vu873u741id/avU+yi4ERYbgF25jd0dJ9/gcCMqSIgWXGzhr0xAfxbI+2pwcLZz5DplGiVDxBCKOXBXgf7X/tMRi9JjBgkfBwx+QEoxq5bThlQSbB9yvuZoQ144zmope/rwY94kS/PRhYWKHSWqHJfqZPmkoSoaZJKZxkOmURS4hqpqjp2oUxgaqgCRFq93IRbXbp0O6bgf17Zdv3xWNc002+7l7NkABGDuFAUS5s/xJQbRh6fSYnY2eNA9vmnSoRB3qVcHWyQfhtlEoPLQj0DFaQI8iSRD0nZtpF9qz7oucw76FlOuGUdmEZqj7p5XjnSkvXJwNdYTBOoAeippTFhuBplRTDq+zBMhEtpB7gOu0klJvIvI5nC6YuKIVEjihsMjDVdk4N9NAgr13tfErSuLBoiN+ykTa1gnADdaSgMPUNPqoeLYXLfy6cB0Qu+NNJy8Db2eFFK9MHSaF+XQ88cnYoB0dZiJzvT46mNYzsGS4tCOC5JNmqqOPA4H7rs1BdoxxJZiEJD2sKDETlZ8JtTZ3rV2+gkdv6clsjzKb/MteDAZq8POJEXqiboBREP3Uudk32LAXWJy5IFEOigCsAwA2aCrBmNx8v6bv/rPXEtCW7HKDuzde5x/NMCKeLSWvUK9rnvGcLM5gqzE4K67UyoSELE3Z2Xezf6WiOs1EyzDMxH+bDAsQNMDfyKpueIpTFdV2BEwYQumYUYej1CfE+DLiNSCu1wbXqMBWR6lCvuEyRa2DTiIFASymxZM4WBYDrxZz9mGT+eBK40HI/gSfjiL5hY7k/+9/9wFb4iCKfFgZCkZJ/rapeBauln+7weGW9Y73cyjhkiv1l5QXJNa2cqHsCkNpAEFmK6wtYJAaLlc7GUOCHQ4Bww4YQ3JNtcJIuEQbUmhvrDm4v1Rw8yklhQdWING3vimVL+5lEoKnzcHTKDHsuuaw32m55jAAGnzEbBYEzHn78vszh9A/L31eVXxLA68hjJizpzd+JnNQD8hV/76wWe/EoIcWdT/D1Ar6iB+pI+pAoF+7OLvjjRc07nFmqcFXWKFGz1kx0LZi+V5QrRxjigqtVleTGx5X6CzRu0LatBI9KKMAJlfa9diqObtXulte5A+iKlzOvmPHi0EIhwn2vRrt1/0Cxh7P2RZ+t8dA415pCc7pit/j1bZ7tjXPJiRGQqiK0nWR5pbossfsmOjCryW1uzlbmxEDCMQ1AExjIc/FPh77mh37x/+wXOLvnXbl4s4Iew4BCugisOfCrOY/O7H/HeBf1UZnXVeBt70DmAHTx6Fwt9Z33k5eDkPBL6CLdY8XJ88gLYO/nqko+5dnr3/47c1PHrlaEvjJMw1M6vcedFWZyjpUayHq1lHj9ZwVFaUEVCWeY2l4jln03BSHB8W95xCnxPply7p4j3U8k/HolyG5wKKWEljjw2oFvhFICoi6B9U95BJnPmjyVlG2Wi350BT2We4QhvUcOnSzEMOsGjFDC8lCCWrzvL12NjAC5KuzAhJ71y+i8jD3eb51F3YktQINAU5AiMgBr2oqWc51rXluORIsAkpIK7EOKPImg7AQNAfojoEuYZsOSjTliTrmaEZdxJHDQD01otLkDpPaqY7bK/R5CbKMQcFxOpjHG9j/FOWpYYgTuF5r9K8kVAqBr49vBmJFOSqSmqIKLz5CRYN+YFjqeoE79CsY8sJfPLsZ4xF2J7A+JagEZWHrBvBSnWK9usaXUyuIJIx7s0ugOJoMyCf7cp25rS82dy5CigQOfgVnn3tzghh4gJjUPdXVX0nVX56vhJt7PbP4NohIFT5oryxd3bvA3mMe+lFFUw9fcuAYJAWgo2HXQwIPQI8mmBPHIzUDItfYetNFueje0QIssDUYTk8RrED2nmd0ewfSgOJfEnhQ/aXPSYAtLisDcJgLwF9+H+JbBHtNB2t57nOh34nW4gWMM+U1XAjexE0EN58Zir+AlLgWlSHRCwE5QyMaNLuei893ek2ys9N5b/S8xX4Qrw4aM1SkrsnHROQnUY88agZGGcchdnKHfWluHzFSRqgydKuFvEMC22S+Ck3oEzegpBhjh+1w98AQ3M0QookQUgqJ1atQwJ4dGBJBdivCdQHGig6cLjAFOlksFgfmCYsK34ZrRQmjYk25ZTrOGjC7yLC6TO+jtWS8ycykktg4HKy30zT57/Svm5EcxT0sEZoo0slIltYD82Y6GNnSFyVrrN/v6eJby9lEfWSyFHZgqe1yQcIQRe7baU1ZS0NvtZ/Ie9huMOlUjfR774A0bbd7p/+kfE4Ol58qCl/oOESRyVrGOn092QliLZ1n3/Uoczq15xZ0GMPtcQm8e9Bj1BgnSbR9EEflH0QvA3EbKe4QsmtRwk8R1ZWg/RTxnK0aYB5Cnk+9nqECFPqI21Pw/ugebk3kQdR9WYsMtk7HGlZ1q3cpWCSmCvbeShgFFSQ1nAwFdtyNLmUpdscmLVUdyXaNn1OVGzmvek3VFvuVyD7q8GgcX+nhe6t63QwTJbl2dbqpifm4wtzXFURW1lv9UtodJ3Z6g0Y8095cm8Mdvbvv6xovtdMosLa0AesLDAeQOuVYFkEloaAEL7zRm01NQuZUY4t0ndf1JnUbF+hS/h9QZgP6HPYDSTMDmgavvP2eUyuvXQDDUfqpp8lG6ya/7U9V6NtDhbHWBEIVQfgWYIW6dju7mJuzT7DYYHWAsaYumzqstyXI0MCP40FIOWWrc7NJf0lwY+pnnfmbot4gvDEIgA7kj1ZeDaVhSC+Tjib7twAkVfO6biMrB0uHAItI0ZtwOLu14Y4/W7cpmh0q8g0ExObR7Y0GI0Qid9/hJUIlging0L31Dpz/+Ir9ot/SVUJrJ167SUWn4B5j8Z2nYIm+70EGoE8jVniMg6UF4zH4AjGOibtomFO9gKTYWYJqHQVeE+Jpw8VbUJ4/7MXb45dYpBm8dkSvTnyCes/1ezS9uAmH/knmBxz2Mftcv1n4S5S7MKcqUbNvxXbo0YqytgN2BBpzvA+B+tjCyE5H6MTbUG7ok2zt2M2dD+L62m/KGBPvWx/UhSDQ298JX8ostzWGOdenp+yf31oS7DrhUUQ/brJSubhE3KeqDmH55tXqxERG+mt/Le5jCdFvjRMcn9z0cUHV5EZ8uozD0Fras3pgMOaSbMZcU61DX+/WTcKDRaFxNXdEgBBfT6LR29xQgGOuQYlbPsD7BMcnT58NqbQgHfzdXxDj+NIa3yvIWAADazq4Fhz7/4DGAqBJBJDZD0lgraoInB94JnNeOXP9AB1MOzXPyrRnCYRcY1l/Yk0GyszxF4jGVxWG4ynPljFniMWnYyy0wxN6Kzdp0YMXpledSc4GwjT1SH0N2n517dD14DghSdr0NDe6q76x+9JGmK76Wveka7ujfj+E3fBKdxq9RO5kajr297p7d28PMoJv12XuzMr03YCNxejMTB86mp1wYfNPwCn8ZyNres1PZUfHZhNAAGNc42dOrTKGMZlOp732qKzdidTf7rMbLmjXZA8iwxkS1hZkd6Indq+Pk6XSaZSyo3UdxdplXlPq7C3aa4P5ycOONlJOxwsDcaKDBnidjSl0vnxPzMEleMFfeNqI86qC8NSBwKPm3ZnxWBi3ZW0fQiXMFcvImEEwyHiCNvHREEN/NcbA/wZiukh3OtDr8Ka90KAD/3dgo386vNE/8z3yQeihvzbsdFc99pjfaTI9JC2LInW7W2/eIZJu0DeWz7htoQQdq+1S7d267bRGnwvFHK09bS5VbOpouhE2+9qBR2CX+0KTt+2nAvg2gAZh4HE7ecZ0QLi3ZzwnbR2Bf1/I3L3Sx6dDmnxAzz5CjV12Dk4PSp5TOe3pFwYEG1/xTX8kn/jTMycOBWHhh/yjX+YrrAIPM6r2rP7oBawFYODKHJ20IYKROVQZfYiDod7FmcMeT+coEBW6wwNoDkFHk7nDJBzlYAlhgzYIUrCCU2dcvia7mzqv2GaAlPDEO+k/1Sg+qzoxnC5szxSOjqWofWJ7BuGBcymzUpLY8J1mpSXIV817URXK8abSVp2nmHqxXsioAtMVsfSho6ldd3gVyTTl/ipdjiVn84hbeyXzut1ZO4a/OOuSkG2bhHjOF+yCsX6QaXHbHunAxbXZ4jhP9Exuh416110rARzvhyZ4cDdaBg4JSh92MjmhfACUmJLZwTrjzmGPzyqhfPoQxPAghM5s2u+pkkbQ9wpp+6nVafu9Gn3JxiAfqYT+qGHwURV6sqko5VvUVkP8FHYbqKSiBidBX0fMAYIhBcPiI8RDW+05bDdIWdtAvONPR2ijVF0ltFnOo/9+lD2Kw0c/Pnr16KcQ0uXH+wCUkrt9+Nlni/YnML5qlpBuurv+wPSasy9RhTklmENv0bvf1u8af2q51c6ZWjF8z/tSf2kES/D1t2x2/Nh9a+EOLPkAkyDAuoHJds/ejj2A9jpdYjp2m1U03krjSHaS1LYTbY5yYeRE+KY/ZvOjotxiD5Kl5ezzpBi+BZTleHpSAIxPsXpX+JioOoNwYcSHLMfuU09CGS5lNvrkTekN+Wgz4f7cY2A8SBvV8FDyQGPZvPuAMiRvGNCJd9e4iTk73hPOdaMMxW4c0YRRVg88gNW2D2FusX8C8z1joJN4/clqAKGKvsJAZVU2+IAK0r0EuvzfRVnMmZERSP7OoZfCMeBHALmMyBUWz7iKpAzM+QBzSDDAwwtWbIzvoHd30tggxmZIGen7dxUEPi6ahDeFTv/8FvqdZyUHWNK+6reLK0sJHkhpAAweM8d/kKUzDHHMN5H+b7J8Ab/ug17WvG+4uAqfn7+4PHt3/py+kXnYzWbIHLEoImJcFR5UxUgU1XlC7R/MknbXn3gDWzL2UWUYgxhTk6u8ABELzELGJ9t1RksJB0wfEElYRoBrwp0L9PZBeB3cehMfFurkqLKODIzO0Y4CYuzZlXPM1rXn5/JSQ+OfgMNeclEWY9XV+I3vL89fPz8Hv+EcEewdHUEQBD51e0THMJqSHlLBdUGXV7vfCQxkjx+x9soXAsd+uSX9OzrSE8jcMWuCG1iWM4H7mE0fHcFgxywa7vS6HTq64+obTVYLwjToG2+KnPamncDaJ1Sc0IO7e024LUXoRnM3SboTNC7xmKLv7pEJrBfWSxp6G9V9b+xj9Q62ZQ4xRkkVc/3GwVSbEMHaS0AmaAuefmO/zAadqu/RQTv/kzt6F1IypZQ8WwXhQGyMFjrADz0Brfd0tQC/OJh6NTv0Qs4Znc+B3JxnwjHnjWCK9vARRAOoMQqRAFJ6PPU7PPiD30WiULC8iW8UFnvySL2c66Oni8XidLcIrZtns1l4/vrlxevz9rD+bIaWsxJ1KOgzZnf8jfIqLZY8ZWbYzHDVUtkpVfTkz3t6HXAaZlpi/w+E4NOk", "blender_rig.py": "eNq9fWtz2ziy6Hf9Ci5Te4dMJEaSH8koUWqdxDPJWSf22p69M+NysSgJkhhTJIek5EdOzm+//QBA8CHbmdl7pnZjkQQajUajH0A3YNv22TQL08KaCettJOKZyCxn1xtaR+dnVmKtgtyK12KTuNYfa2HlwhJfxHRdBFYextZGxEUQB6NOx4L/JrJ6b2L1Cqtv9U7UKz8LF156a/V68D+ot0pmIkq8RQQle8m6sKZBlgoAeuF53mWn8yGYihHAHHjWx1WaZPAlCgCXCP4txE2xzsK7YBZ08e06FvQ3TWKBnUhDwDG2RGTla2ikS6hZU8A0gyrWrUWwggiAQCUxC+GlM/Be9q0VwMgAxFxMi8T1oOLQs96LAh6xTm6lWQIlpiG0lEO5yJquRZYmljMNoJOAzjJZTbIk7wKqMf4BXHKXEViFsxBokRCqACeU/ekCRtNMYD8skQONI1Ek1jSJrRiBUUPWp/AmWCUMyFnRA1B09CFMoREgmWulAfSGRijMNkFM6AZxuAo0thKIpMcvcVjcduEPNB0BCnLosdM7nnUi8iQf4WhPg2i6jgBgngAygGQAuCHymSD0iaJTkRFtnU1yIyIXsedWgEQr6A70ekagiLyazW6xARiVOJ+HIoMxM0YZ0fKsgzxkQHFizfF9TEMULNZfBNAZm8Z2l2vA11oBm0L/19ZdQliu0ghZyoG6eRJJ6mEHJkGMvAMlEqBbHkyAHeICRsqzzkIrgIEI8pDbgyENbgGZPA0i6AU0zmBmYQ6MPwU6AM6ACGGQI/V2PesdDqdJfCcTaZIn0GSwCuMgw7azTMDfPIjWs4B/FPh3EoRRkBHz7XnW4Q0z/89HbxngDID89PbXrvX26PDze2A84BOeEDDeYpUGwAIFkCR3u9bx2/+Sg312fkS8eIc8DPQjbLvW0fH73IrChSCW3YQ5fLoWE6DqJpwJ7s6+Z50KYq6NyIpwCr3BiZ8iwZEwjN90DQQEMugyPBbl4Hgd27Y7Ic1lK8gWADEX6vlLDvwif6+CYql+J7n6ld/qn0W4Eh31MElv1c94vQIBg9Iq7cyzZEWQ1kUY5ZYs8SkosvCma/1rHRQii2Fouta/capnnc553xoTaA//cdxOpwNyAIZ24TwN3BGRMc3CuHDm9gVMPeurUdjqWef90b43mH/LL+2u9RTYZh6t8+X4PFsLhPUExNxf/A+plnfgnw1gCvTw8OeF+uGFMKNuHLvXs13rmTUYXVrh3MJHC0ZLlYL5CDPu4rITpABEDYN3kC3WK5CPJ/iUQeeD1AtmMz+Q7xFsGEPHZiIvxnYYp/A7E3+sw0zMZB9bqoBcf1Q5ICfBngfrCMAv1yg8gQFtyb3Gf9Nlguw1vjBLWTaw3yxbgyxP8CkO48U6TuzL1taWIlwsEbHiNhXjeZQERdk4qIF2FJMCeGZLpb7Xb2K6FFE6thdhllgwPWH6gHhbgJCESRJEQA7AlWTh73ZrgzQBTaq8Y8lxT2kfpJTCEBjV6FT/Zb+1Wh6giGyvtNtOCBA0oPFBq7e31FonSpK0vZGd1vLzIC9ai/e7kqwpsNIkAAVFHUdhWiRIGZghyNepR3yNQHMH+d7tgDBbBVeg7DN6k3vAnF0YGRDjfnIlefPj55+Oof5XGzlyRBPOg5/AUgEKxxzeXVx+63RyMCYEFATp44EARvnr0bvOJqq93oTi2o+CW5FxLS9jNTtPEdOd/l+WDUQRkeUkrmB+gvkwLXxQXY6UWnPgs4SEALWfTMB8K/KRZtjEk3VyUTg/BSAglPALpkW4AbZ3QFUkxRIaGTuuhFptSTd0gw1x2bKFG7MFpnSj4fL1JlI4eoQAEjqRGM2n62wjcj+ZO/BNogKK5R3o+qAIgS3JcFL2BVqxWBAkvSeruqC0dr3dZ8/3vBs020D9ux6qJoRUZLcl1plAXgf1mBcIQwOgAuJmKsBkPihAo0zWhTjMsiQr6wLUnPsIpu0YBa4iELECEmkBahpqI2jgL3qNc+Ti0i3BYIUcWkixAhXx6NGgrSo2nZhAqRTKxWUQxyKaBIsGbD38gOGzMXdzOqn2UpIAikj6o3rP/Th1YIQkrBWNz+SLNwuKgN5ME3gTpx5YI8WtA0PhgKJUhoFrPbV2YEbT5IZCJEr3d10JS5fzoFMimC596JJjT1EUThO3hD9NYCblyyAVTm/QtXb40yduOQBr9BZxhDmPKt+/TrJoVukTAPmH9eliBLiMdi69c9CZ/LRz+denJNsbQdZBSQCyz7teAbbBzJ8HaG3cIscXoKRyZw1SiugkJdB/UrJAIRyaHIRhsfTyNAqxHks/UOHuxeDSi5JrUvhgK2BxYCHHRq8M1Sj8Lea2HGbVE+6azy1iAWceRgJbGGvAHRFJeOMxgIFxsKtAgBzw0mdYDwOYT27se7CAz+0wcsG1siAEq+fsNi/E6vAmBH4C5gLOSCzyDGisZsnIsoEJoFEQfytgLYFkvEi2yE+0rhIPuZhw/HR49sG+RDLGCdiwVH1b4+ATBdl0GW6o/SLEESefJwcVplszYZSC2K2QgXFBZQea059GIsgcmlj2u6PDg1P/n4eHJ/756cHns5+OTz+hglSAGPpF/7IrG7oAi5HYgOcrvnKtN2BHtjX4JQljYBp4AhI15DW+9+KABIP9iRx9u+wXCZoKKSWHIUGtMCcCwpdSUmHjKFx0M+B5JhvSSrPEX8dRGCsFXhIK5FOnhjU7mjDyfpCmEUimZBqgaqC6YKeiiVc+5uCtCmW/P7E+I8dE4V2QjeQaQ1ctKSTs5qOB9zuaKJWVBaBvCO2C6xhYvd+QwGxVkD3JfVTCs0TQYU/FO5UoOejKeNBUCAUcAwA4ebswU8GGdDskFisCurMCB2d1w9ISrEen73bpZ3ADPztL+IA/VzcXw0twYVYx/O1aA9HbB0HE/kHusblsPbeWnSn4TtNb+OBA0f4lzBao2r904eOwSy8H8uWAX3a2d+0MyeuA17nroiDml+dYKuIuOz1srje9hX8IM9d1S3jrdIbD3Bhj0Agz4eerBGwQ8MskP+tqaRLdLpIYWPu1tdvH/9gj+gxeehsBP8Ab0lDgZ1+MutbwkojnsqmosFA6a1xpS2s89PrBOx+3Y1L1HJh3/NU4S9bxzPmAms3toBdqk4gAuYwtX9gKug0jVj6or9QgfeJfxMKf0OsNgwgM21wsoJlMrehtBLkn725BhOQwcIfguwv4SysCIqfJm0cJ6QepVQmSjy+ljFqhVwqP+qOa1CsL2N9aeajqYpAGedW+iUnYefjFLzIhvFoZCSUuZe3bs/c/+SenHz+/+3hydPjebto0FTvO/C9G1bAu8gv7NFkvlrHIgUqedC/8TRCtUWj1vf29RnVp8R3SH2DQ9gbI8PtT9gNIlUJMcb2IlspoXVHa9Lj+6Ct/Fzixa32QMhNmCMyPO5rhyJ8oz/nXQP8asvH5xDoEaSVAUmVxMLLSdVzw4i6wXEJLOrzuJtDStOYwE78EUsJF1iT4kiBZPhAoNkmpTQetrUnu3OCMAj94uAf23QfX+j+Wc0dv9ofGizfwYsAvGKkpiLHpUk8yAqzmGaoYPYPpi8uTte/tvkQYisFQYTjwkgBbr8cK6mscyT1GqBwt3aQCoz48sVbrHJfyviBpwLcsF3stIn4AqhIkMEFahqlPZGdoz7Dvu4SUJPYJaAQ2+oUomFZMkP4+FuPuR3PsOhS4oH9oAJFI/DVr+/pafZ3f+JEmHIwC4RY70ZwLlqSL5ppu/R813aB61lI9a1TPVPVepX6RCP9WA1AIDurjhh/K+jz2TEAhoi0ASMC2AID2d0wSH8TTJa0r44I6sXVlZZ8/8MI8GzEoh5hrkRXl0PX0mGiu5S+SRSOx8G+IXfov90y2Q+wQpMFbE7LR4d0F/aPH0+sPykE3q0c1h0w1Vh+WSaSGpYIT0hrJRc9dbIe5HX8OiB1dRazzJMvBxA2IZrw7kgr4zdMQrL3CIE3fe9GvzeMXL/WL6pQf7hvzOV9W0cdyBFyirwb3KVR82VdkoBJ6kAcDTWUJTnUTH6mXL8tecuuq9B0N1MvBnsGn0G+D0UqaarwGBrdXUekr6n3C3ZwREU3LzUh8gbdWwIvcEdiBFkpLkk/9D9Yt/P1x74NcLJlb4InPnDycCVMWVWi+W6f5j6UwxYrwcAMFiSjPSq5tcNUUB2bYr3KWdHt5udtR8Jw6sK4iGBJ3jxGCl7QsVLYDaONKOHnaCxoZBjeVLI8AaQymLMzBlPggWZfGhLXHRViiHoPrQu8AJtjzQbQAOwCsxSnMzwKXMoKbMB8PpJbZq84mhIYAYGQDNHI7WzrNHJCiydq11MPAfCArs2uh1c9Dv4y6VnKF05rGb8Cwlxm+zdTbnnytdBFWQFkDRYzBZstMLiReeuCDgL/u2JMsuAOlk4qFXBuWup8239Z5kIH6sVttjfI/u7IvKXBnakZLxQYRZuG0cD6MwaJk8TbmP11WZWP6t0taZYz/0M8Mf2bNtW0S/mP6tysl+Zj/dFk2jaVEQvYa07RtwMDZip/uNMuNNeshVQEPJD79zMbLDBcl3x5/Pjyj9bQSzhPc38N9UjSzJ+KO9vVw3y4NZpleylTWkz9BEjlKaH/AVWYAOLuwP6C5DH8lEjaz1wn6sOV+Kw/E8o6rEM1kwRyZ8GIJvYGvz7Su1Y+Dl5XH4YuShWMxvVKSa99QkMFMvf2xlGdvsRlNTMfGHWAw+50+9oQ6wONq8yxkxeOWBXLJ8ejxGL6HY5+lYSzsZkHjeYjP3GKz6sBuljWed6guN9KsPLSbhdUzk0fXHlSrf4avdkth9YLJqGsPq7U/wFe7pbB6ATJY6RpuqVH7PEn9w9gEYtRRrwZeX0GhFiWUclUY5ScgH6wEczwtxDkDKH8k5rgXgtxGk8Im0uCSp30q97HUtwy+GfImRyO3Ju+hKM5GZHXJLTgD7UtDOE+jYNOsqGyvWj2Tx0oQ11lI5kSOhrFD/enBg1T7+7qciCbJtS7HtcqCe/BSYdEntc9qHX4abb3FxeuLinRxaO3pGYz4MllHYP7h2jH0CsVRKx9UKh1kK5tLEno8KhVo22r+BL4y15Y1qUclAPy2re6HgHhI1iCSlaSjt0yUnRKcas4AWarE+Q2LKBTlNm0F00CCmcG2DX/K7LLGkpRyk19IkiuGAelF9vLOXm3Ig/gqEkb9+Q3zJSkF+5Jr7crhU2ZqWf0qFljbQSSeMTBaQKqyQG8LD0zAcW22TepJsei+bnpQbxvKba3cbWPv+1jul/RI4Gb2Em0W7FWLwKxU4OJckvpdDjDD2s5tCU5+WQdJUNa8r955It4GOUp6rgM9NXkKgDYYSloQF84JlNExE87S1T8LFEr4NSVLOi1X1Vxe5AGKQFOglFGyvb2UWvkPtW0v1bK5tIG7enpXv7KIOkIFv17FqOnXIoqSqubHDT5Q/kWQ6329yhpJY2XkFljrdqA9hFvpwnbLZ3Za2Nq4xUCROS64C9w/xHaZNY7gPYDpQQl2N5PZrXL8ybw3rNbJXZuTBxUYrdIfwXfaHdkvzYCb3ASQigxXasJIaKdLLTeQ1L6U7lfXetFn2I7x1fWC+NYp/S94d1TaGnKhZhnAwMEj9VCaM0ddSz6SPCgf99Rjn40dKiyNHeA02lhRlos2PEozQonoSxl1I1Ah0RKt0pdhl9cNRbxegXtfCGLv3NB+gTGnYXwJf3A4gOvvTMnRWgrjZ0hm0i8gFHuG8E0SSfWLe1mB+FYZ9g7wPHDkpEsdMErI/sTaymMmuYV2JFwSUZJey5JTsKxaHENHp+QS/KLHz1ilKJFRdpLR2WYn2j4P+POSPmvd6dbBS0PqgfqVzxooG1YNkOdBGNVAMnf1X25DeYAvmT+V5yqlr1sztfK4ZmLVrCq3tgs/BxEyj9GpyLiiop7kdPsn8NMK2zQvLEfiKylsvw1oCBRGVf+8SFrU703epZZlpxpqU4aVbq+oZni1yiQp7qvSH7bUKQdmbn/N429f5/E31l2AOW5yzYg4j6h1tqRQMqoBmECrFYBuxWt9+9c34wPc6l1nogM/lBzR23DhwvrbuIwY0zuR+ituOJShZuWYvUf7qnVlvhQF0lut+Z7v3dqSgOm8g70ytr5ejSyHN4A2uAFEoigPY4o4FfiOBIKc8bShugpSJwpWk1lg/TGyuO4fWLdrbdzaoLT9hzx+BWWRud97YSFWueN+41AXvY9udqpddfOiSYA7XDJagTa71BDkXiyuHftQRTnLFQoemMY2b61sl+HKtT7aQAZ1jwFEGN6jKuFmsAMA3erGuX5T2y/EOHSKP8IfY/vw/cdziZRA3fD1m5YbaMIAxxaRFgJEilFFlxCGHqjzgslCfYgNfeOhqAa+9wqQbsgbBNJcwXPkt54s7Hog4RfFEtTDQPR2q1JDw+GyNY+JrVdT3XlZQqZyv/TBJhcx8tzk8R1FAy/IqogQGBmIANDgEX5fthTBXcBpEoOfjuXKdcX7x+X47X8dvlMjw0SehXkaBbc+bwuC3D3/+O6ftuInL18m134Y+3OUy/CdlvTKOcfh2jTbUIFSD+XaOu61CpPrdBRgWbJrKQh6gf1PyyeMpM71MnF+BUhPbn0ZXy6cAtoXhaEtwKg9leHoI65s5etgIziy24xLp4XpXCwwrpI2SjgSW1nFan1Lb3nLpspFWbSpugSSLMGusgYVq1DMGVKbRoQ4vrlxC8pmQsMO3UtgpmM9Cg/xQDHk12GxdGxaQmkJTsNYpzBei+oeuRnkxYzvYyxHVBVz02oxnChtxQKct7jMHFT1MdeeRmHqOODawHcMVAgm6JeiQwD18JHiJX50abYNqpD/QIEAU5I2GVDtXIJeDSaVMkRipSSZLFUgSHz1vbY2jkj9oVfGy2pyvKA0sML0ypnRmoKB3LWMz7tJnR59hS45hldvPX1qDY3iF1TmDeOCzhHYSVdCpLNwlXOEDNQbeC/Z1Sbr4bIiZK6t52Pr2svXq5aqz5CCg2GFsb48aN9TpOgCY/aJaSnaQtz4C1B9IEUwaBCMdYDU+pmkMgAcx1Vih7MbJk2cxHciS5xrHLovag/PvehfNgIvQ8QTKjaZd7PAyGbnArdKQvdS+ZPXFyHCRBlyenhydPDusBQi7yizRmXTUNaFzL/JKMuEpzuGuFIujfSPkptbGYM5TdJbGY1Lrz2piXXYSaOAitM6wSf7EbqVqtWb0EEfucdRaDX1a1QiXc81QbaHmPgjVf2pwLgzm+hC8XSqAmkDRPLfx78eHtnqLeUa+Xl4h98+AA8PXuKWXTOQdwXjEKSASkiL/arQ9ogOHb7b1EqEsAwgU49jaKGcudInM4hTRh29sQZDjDoqm5ptJcd7MUVavD989/HTwfmhbUzwlZdhkBTUZHDQ921NVqLoHtGVmdkV0oXMXxjzRgqvu7UpM9qobnyB6qAql65yezHs3tTLlSFrD3BEe4DDGw9OgSS/nB76B7+cH9tu63BiKpBhuJZNVjcz27fl5jZPNVSpnLZG2WBgTIysr+KbbHIewhxB94skG4mDBYoDpk5F5mjekHiUiD1BBQzEn8p4rDjBlLOWdKtSL2+k2WKMgo42Ky07DCuu4LYpcavUUWp645m4SnxVH2HeoDxlqG+UoI83Vbm3jcr3UforwfzWkl+mBd2rB7dAyTKiHdOkkq1XGkR2ZZO2MQx68MifahvBak9bSqjA1EVJwIY5Z0hC6f1pQ9RmRnNk9oJrG+zCrp5ddsaYn5SqqUP/JLwy8IMiL4HDsnp2ZedRZhzgUDHfcEtChui2KF1pwlRJ1SyqFbBp8lSDd0nQFVJ31UTjiZRE9vuD8wMZ4Hx4aqtKUmwowareYicQDV8qRS164Bu+IlcipyI5ZQH9++fT419O/P97+PHnD+dn9jdVmICAx52G8YJ05/HRbx8/nx+envifDw9OD8/ObVWU8zr8DfXbV/kuGdqm9sHR0f3lZmTH2Z8PPh3a36OOAF5J1W3R00SaZgS15Joy3Nm6hRmxCsGOAv7ZlXmuNLPkfLUcSiV+zpnEz39OZkmxbUDv9/KY0P7JARBTDuZ9WsFkKRXyW6EfQ0Uyg+1FmRfjvreHGYkpzKrx0P2TZkClYSKOXyRFEG1vnQqNd90HocWS7oLyqrbDS6ZXPkfbj2XK1nc60VVx15ihbkUAVGUDTfQFsxgG7KNr89CELyWjZDDOKc5KoY+qD/4/o4AXrQGcXL0Ngxh+hQKzjG8lEJEXYcaflGRzR5yeHhv2MibqK2ZFlUKQV0EuoYC5PQ3ihJLNMcRrAAbV32X4IEvLPFwAdkFU5ud7TMRKmrF3NcMAZpVt/M/35/C0JUdqEQ6l5f11wam7I4tJWo5Mg4osgK4pIJ/MjguZ9ltqdyODqqEwcSDjW0d4DAGNUdHbpboC62r1L3dqlJVhZm+xgwA6jWg4xkI9KqSw0kq2b70uC77Ggqg5VLGSX66wI0wrpwLI7TTcrGZthgBUyAFDE9OL8NLDNcnQ2GSeeZMgIi1sOLc5hefiLKl2ge27G96sqi571CndWPNQw4IoA+RHrmv4XfR7fWgPMJ2HuPrrqZy3CkZ6UV22A8a1IxnqQvAggrepxtl9hOVklVxg0vDLpWSJy/podKFjWJrwaWYjxvCde9+ScFhna1wWZEcZsL6uuMWlmaRw8hVP+RlJkJk0ntDJVp9c7U1PoBbl4qgzDyxHHWWAE3wdY4pDEdxhADw+cZlYHSURg98ocw5pEfErT9ERMPdDs/R+PmnMuQYLGbMUN35aWCguLuqDfomRCYM2qgEBfOocEQvqcqotnk9BDB6rYZsiVvBd7QEgLlO2/pXlFzcW7i7VpGeAD8Y8SiMCpTwe1cGZd5RySJllDMV1azaupUccrV012Eb6yytlnUg0pK+qwelsK71ODYqNLZVVu6F5IDcuEDflc6pV6FVpZiKYv75PVZ4X0jn4/PHTGe8/cKREOnEMw/oxG1RqYccIXsToDXhr7q+hZk+TXBianU1HbvY6S/hF13oKP/MyNgNT02SyCogqPEIHg8q/yGNxVmsM1ui9wXNB5FEbFq290tfqMjSuv6ruqd5NmnaFRFiHAXyiXQ9EWyX5UgNekfh/6EaloD+Fsv+qv0SGx7VTjKGhmZwlZlp8rY7cV8EKLtXA1eBTk5SfQBiTIJg59A3T2iQZJ+swmvmciC7JOc94cR2J7+POcl6IFExSnfBZ2T+jmqXbpM1r8mnmwZXw4Udm+jQ4sMRRWJM8Hh8PGSqU7mt+l60gO0+Lpm9Ycsmosp/EGYq+WqL71y8H4Ah9/nj8mUf3StzyCglwVkn3ubY8wTtdCNytUgRBOri4KnzBby7L5pAmzhxsM/6CK/H8q3S3OSEYX5LhO3fb9yvaOlPrUMlEbcxTVlBZpOWOuhHqAc0gd/MYO0UNG+aEDcwKNHcRJSl0a0jxBJk0NwfMmYKLMwSqzSDfanPUdpaDyDwdp2VDZhuBNkFU3eyrbBq3Usus8sjBAWbisZX2nt2CjC1ZaTxv0koJ/q00amlCYbwNLjE42S/mWgYxOenS1iMrWpMU59O6+nn327ujwzNj5ff+JETtr1ZA09kV1CWaaziH9HR7tAdMCukC+4f2Ax9aYYJQ6mRa1APuUchg0AMDPaOcYzyOIOQAqDRUb9JQZ1wsUZS9L0Oeu/pBRazmSzPi830Z1tynspWo5icortdRgtitpQqirAtaKEQQvDK2BC0J5gg48qi5JsEXWvL4H6v3YihP7pHxEGJDqXeENQjOeOgsI++OQpfhb5e2B+HXjXxzQ3uEOxIbqp7Vqmf16j141V5/llzHsvkb3LTrKoSeWZUs8RdDs0LWqJBtrfBr1/qta/2O4zzgKAIZyj4of5a7nR+WPFZ4/IHexYZpnDtgIsYLP8KUeP6Z4U8812OMMW5dPheJC/BPKOA2NP7XCo9TtBaGPY/YONHPgNdvXUWdnoKN6P6q2o/cWlAsB3uZ0PQLBtdzJPUUwMytQMwaEBEdFZltolgGh2PtHlKhHZt65crLLbW/lZSPBB55hPk7gGzXusJfV/hrniR0UtWjCMyhxyb+KrAZ2w+2UbJazXil6rXTq95WWeNqW0uNdsw620YlKWpDQhHURFFMN3iKzcGsQEptGxsDRPmmBiNrgWGM0DJMc3+dOrM719wEwqhAK8D0QpBEAQZwPPt9hIcZglEtbeffLMzuimBqaht7W6obTKnZHUfJ64YLVKxO0cIABnbhLKqWQevjzMG9HhDVTy3DhuEj5GCiy+SQCUfoy5+Y7WuUvVDhvag/mIBl2hBTjzaUJm6lEsWVllVUvs7vOovARA1DCdxafYy1xPqK6lCN8xCW1FidFGlJiesguqpQAvcMysYqZKBZ15M56GdOQLmp5UNzBabvvUDsSSR3sRDi7u27VLH1EwbXh5XeqeMn5Bjs7Om2e5UnjFJ9mCqcm72kEJXcYbIGrsuBpxROu2wOZ300y7F5KZt3m5WG9VpDWa3Hcbu1as2Rydbx9w7MC5Mc5UNzYAYUMVsj/i6OS/uXB4dl3xyV8gGgPTwmO/cMyc53DAnPr8FLVw3Po8h8HWzEY+nMnZUxh4NhqdCl/qzZGvs1hqwoubIDdd33m5SyPZVOoWnySKGBVNhzNaT+nvtYEfVbV2UwPUi3L+tVWqHbE6uPM9IKFsF0iQcTUmA2cqUVhBk9vegNrGlAR+JmYrpORRZk5g5Mwekao0aEmzKmHQzi4nBvGJ7nZvwWnXqLg6RPYECWmla9GIwloUj16nvKylYBM2y+amxeVLFZo8FYcDqIS8jsNlGQgokw0KivCWm3gRBAwBx1jKKvl3WbWFYYbLDbd+9roumfGui/YPR3thObcFq7LXhrQt9H5/4WOreIritK+9sHdtW/ruh/2p7bJnnkZByWdjYeL6zmZSVUv038rNPvkS2UB3GFMVje8N7JMeOg1u+T3sOK4HVZlhove/y2Tc3uNapW3vXqOqquVA0hw3q18gJ6292yvaKdG+9Hiovc1aJf+zrwpWd8eVgf6MmrMGBVMPwOVfBbV53BI8khFcJeOxLtkvQ3Q1c3BHB1zCtLn/YpHYMNIF4AW6KpKdc95fGN1cLqxNuutQOl0RxrLUUHaUOhIQwnWAZtZc74kG0otPuyS3ptS6mCCwEkFOJthd7S8dwSEjFzW6kTXPE7R6S6bHbLfg7cRuYjr488ennk11bPnN3xmukKDtcqHaPEBYGBkQ+PmHRLCnwh74gSmkYImvUwpS+NVM4TV64zv9TYsip8Z9dLVk3DMmeK63+rTPev3zq1TUQ6MTBdtq+LphexSnbVLBqXObXkca5S5mt4ndTzYGR9ykeqApApSiTbrupGX0KGH1gO1alC6WrlVJHZazy7dr5ncv3aVQnLD06uVodty9AqX48xG7WhOSyFQFOqcZqeqqeOXfjVcMS21KN+jdocuIEeG3Tbvv0nBcb+IwWGzJkqRnrOgPqnAxvQc+Y8sZU6lbJcM/7evKKT47PDMv7wwb27xuJpSyJWQ4J8bwSQ3Jstdx05GwY3ZWi119h+NQrhpmq5U8sFlVCDhgiZxlblytykQEpSvcY6xIuhjtDEzcuYDxyslVX7mmqw6Zj78kGW53jJGM+7BdmRaUTl4GF6wdhc1H70ntgT43Dp3Wd4lGK4iDE6P7CyIC7PHQvoKL/manxIx2p7dHAi7ahvbdTnExfbtioeqDMum1AZC49Y4peDEP/1nWxxI48/Pv7lXB0jimdI4yG9ubmrzQWdqzCeqfTaYikZp0K1ubnvRmAusNKlcb7xJMgFDqVDMHRhTue6kWf8Aue2l39UCPnWAGbE5ZsZGK5bPjw9PT5VBIkJAe6tvBUCDQXkmJnIp+C3JJzzoQiIi314mlMRThOwrcNVmokc2WqbXNIzUFkgNDv07/snh5lZWW6h9svZHfmMmC9DSZ3tZ8FjLEXr6e7bMNd5e22Vth0Lr8I3qpKx9QD5yRfoRmrwC8kvYFCQZXwZkS/vV6Hjr1WnfXhQ3WwjAO9xY0IVna+lj6FOu3IQfT5qemz/fPQWBh/367mL+qThTt1voGoU0suxnRrU7TqVhxPLFwUwbJJks/w+SBxSmldr6nydsX3468nx6bn98PBcXSvnSOGoRFANOAa9y+yu7budSlOpOuWB4k+fXl1rWKWYk/Gv784/Hn8+s7uPimqzjFGYAuXxvoswXtRwK6VffaP1/DatH+3fGhjc3glkIinibOAjxWDABshWIO2gQOrPJzf3siXenIUnoGuWxPPOH2LJVsSwYkWKMqsiwFbOtPBijkgEc05mltxoHLtzJahP4xrLdMtPGJeCYkFaX3VGrZaLo8Dn+w0U45ff5QCCUCxA6PpX4laNY9dQZcVSssm745Pf8CCm1QQMJL4fSSguVdnodBox5xN9/gXDqktBdBPm2OJ1kM3Gdu93m7MswQsf2wgW89RkxLnPAH7CeEUDkwxGOrsluvmUoEn1cjFN8GS+yodfbZNPcKS7Fg8JjYw8upmO5mVVEOBxkMZdYfLuL77dq6ImUp+PaN/OXEq70CH9msPg4c/I9m2Ct+Xof9ldgwnhbSlACDqMnATXKt5aZnkp1XikGwxxDwgSuL4OwFAcKLlAjtXnw58Pzj/++9D//V5Y61QPujm2SOOuxT2lDtP8z4vo3iHSw7kz86BsOUrw8J8epdZ0CxgyaKplyHI8I+DPDtk9NF9EyQQPF6cT+Af9vl7lfODWlZrQw1mKmDdQflgJN/9rxakcWiTQCvcouAVqiGbuCV26NqJr5+gCNiMWeZag10IHSamo5Od4dgQYflgoRVmXu1uNvdRP79o5h296U4aMWinw+bW5jMIAqonD/P6+zGFZQqcOE1i7FoS3NX/4rgKkjHgyE4flRx2nq6M+dTaFxFC/XOiDpeppFFCyEp9dxvID/TndG3O988ZNNcYtNWFc7Azd700CaAvtJlxlMI4qgSnxt2O5FHEzsm62Ru4DxirnA32fBQMwKLMlEZssHEpWg87QSWorfeo9vHd2GuH9tfT7RVv+feXUlrJhCl0jXvMp5JivnnHske1e9AaXzVMaAAOJFCBCuxwv9uRKYbU9D3hlzieIYKbk2HKeThvHLhg3DZgBsVpI5HRaMPg+4HzTMuOs9QYCWoaYbb91oBYHmuubBfB0PesdIthytUArwuaoSadyVR0QzLGW/KbSrWtHIRDrE3tcGrm+UFYzaPXoAvqk73HgdBXOoqDznEFjbHAGIGNugJeSdVyYxz3qxP67+zLMtxytYxZmIV0qL7YTx/bZ4VG5VNVSAQUmtM/WF+bHn348OLIfi0xlIYxNR076fOC+Hzq2hIxQmf8gZZ9bTRDiZWss1ZAKSW22tOTHKMFaL3qRVBPuL/qX9REsG1CF5Klwl56O/C/bqabo1eVFi03xUP+S73PfCYJ5psZ2d8pQ4eldu2/9CIewzcep+daNw7l4gNVBQYTy44jxHdcWdUob1lEtdpW+7pKiJ2Pi6Pj9CK2EnA/6plPo8fZW+OusQnBIjAuEb82EEnYRAMKXtVjgXa98y+tURHi0iMg9tiijZObQyRKVlbgoaRoJUXKfgRBpHp7bgLP/FeUKH1nxFNMr3W+POWkkSmppxPrFCpPkLlZE/hWlEiSlIUF3zpRyW6ftlKctUxUAMjLW6gxDRI6TFMF0QEfl+z2ncxgnc9Bf9VLqrChIc9QxIQUkQ5dMBfVnDui4dzVtNWzF/J6cJlmtmti0RRZEyf/iAl/bChv887+yyLZtqayxrL+t139mtWvritp3rnttWz67LxLmoUUtXj2KgRVuq2vjMLaFNHgwb2Lm83EwXevrN/eifd1dydUoue/cmjYxGrXJ0Y4fzyXE5k1bdI0VhkL7RbDgUzzpOABEdeDv9enMTo5Mx1dDf7Cnj+mE8UXQT6G+9WZsDfbMM3z8tN2DnysX/iu2+E36Y9URcvCT3ofMxohfCuwNMEdSINMLF/UBu5T/Vvd0j6wAr2SnW7q8ZbGKLIfuLpJXkmN7eN2pQB1Be1JKIUhxTxWVO0j1x5ja9/pvs2RK8hPfvXlNXyIQWWNb5DY8i2D25vUKr1bncDaYeOti3ntpv+nwazraw8YLKrGLNqUegSM3tq/DGfRtJjZgkPbooRvGYQGzqyc9a4RRhEUk3nA3D06snfevn/Orzut8moVpwVcDI3XXkbCtPJuO7WVRgK/1/Pl0FntfciB7uMlA4BXP43T1/B+LJFlE4jmNRw8RE9k/drw9r/8cj1SpvMcTzgACdPQ5t4bNFrfQPB7G/HUVZIswHvVfzaFTvXmwCmEa5HS9Y28dvpoE06sFnb85ejLYH7wczF6RszJ6IoT41jEb+koEGIE+3Fy/4mv2Ri+Hm2UFxnA43BtOv3Vg3mRf5ZmHo3kkbl4tgnT0Mr15hQ+96wye8J9XaTCbhfECwKY33zqTdVEk8dcKxPn+XGis5vP5q0mSzUQGXeIfPQyiW+cEXEGD39ZgF1vDXuOZW6PBXtmAl8znlUZ2d3e/Af2IbK+fM8cg+ZBDDBJY4QyGcSOHsGUfxpoG6IH2kIOyJMqtYF0kSAML4yPxtr/kuhcid+VhcTu2BzbNqxz0Gzx4+BhvwiyJ8QzEHsBdwPtYrPFWMxxhExnAbRZu8KqAHGT9hLaTAT3aXsey8PFNawmYpbqAYtE3HUA5L6zVZgyzia62xuzaw0jgz7e3H2fOD6vND273YPt3ahmKHG0vgk3/4L6SjeHT2PfB5Drz/Ved+Trm7etJAZZrt+jO3a9ccFJC5KxQCdT5gccTQE48XDZ/J2duAc9JPI3C6dV4/iqVbuq7ZRjNnIn7Su4dT4DBN5gLd7iBSkcws0B1ZIhlMPuh67jjN18PwE+Gdx/OPx2Nf/jhlYPlNxhTNYnEgdZ+//3fF5cuXgF8GEyXTjx+06E+HHRjEP8w/FPh/OD/0P3BAuoQWASjan+mk/1ewStkFMf95sL/XnWQOBqkczHvFpdQE8EeAWkUFGTEOdR41dHT//Vz4lxgYxSHKm0ZwaEVetHGtl0MSYjCWTAD2Yd3rBeBfVkaoFddi+/adbTuwUgpNOUTa6//d9JEWgfBp0/rW7b0E1A9f68cHg2a6YpPYojqi1DUY7mgcCF3zfEiB7kEgx6slaSoK5tLiKVSQWTsa9vFzfD50tiTX3rXWQhmORbSw2Ir9gO0v+QgGWbrVZo7iIrrmium1IAPWgw9HXroWg/gQdrvQF6dOxM6EgPPdCkSvH9rlQbg/+BOfyf1J/j5vqV1j0qUC+r06LTFHyiTCIkIwPDwHW0If8f5QbiSHmyEH+RgVYYxQjM9W2q/SxwDPpkyAdGv0vbNX43MQGMuoJs9ZyJRQTvr1Oe7o/W2onmftIhB33HIDaejSo9gQ69Ofmm5Oh3Qn+e166rxnQCQeLc3SAhcv7CndOGofWl+rPjW8kbq45Pzj7/SpPrl/UH9nNjWyz4JAw8puS7w6DI0ONRRwcWW0iBbZcnccbednlOuCBaE4YzP1MP6sq7bfjNoe+HRVnN+hk4jej1lg61leSDav4F1GFx1vu8W00qa+CN4W/rwNJKyV8gYPwNjIM0Qvb9JRpGH2GmeqVSlvXmhrx2Wj81yfBRdnGDa+mLrAZI1rLA4nY8gWUlhNi7fMG7HJ4efwaH/+fD94efjj2fy3LVHk0FOGZi9SbQmt+tGdYhmnJ+J+4vfPqI4SIyVTxcq6/2R8rBFLkl3w6tL5vlJ38BMR88tmAbyR2xcZxN60DKwAK4YlCcvtYGdRH4Wg3+VgQ7JihC0i11tCSY27iD4FL17KSn8E2AfTm0jBJG7RUtsMCb6ykWd3nhtbjNcJ1k0kysZVMQ2w5boK0bUcsX2nYAJss11fb2fzj6hOxrYhpVwJwu1sN9vWc3HlKA9DoHlqN09vbpfVhw0Kw5kRs0T62g9pTNTrB2+WjIvz2Gn0yGiZNoFGxbPu7kT0mZ4F4ESwSjaIbqvvZ2uBa4MaOkf+/2uNcSfDy4SYGRtBOoGg1EchNAbQlWAsbPXp2sDHgEBbcMskDef7SAqeN3APiFhWihH5vhF6OaUJ5nQGtThgeEWH3loMy5wIogY03/oZF/vJZ9KbRSTB//iH/022XaPALd1VDaTtBzcAW9cs0WjsD5oQoDHSTtP1YP2P/CVZZScV4J28TQaoBHYC3hAhcOhJBgVgB8IlKFp7l8N1adana3BdOk0tkQw5CTEZTMfDDEMLpnhdaN3YoyoUaizuu5epQapS1HQ7W8svslPepOXWrU7D237cTEGvGUj7j+7AffojTe8kZGCzlWG4qAB4P47wV/uKZrUOl7dsntivQtWgbxyGLxXf2aSi91ZdbwHFZQdoaJ41wOSZ6+v3m29GENW7nLFx9yKAQUrxQgVPA0sWJkBwPjIBnEQrrASHXfE1wB0LZW0Du9bZpCjM+TohKSnslKPXk+TvPpa3ZuEsL5ngpltP26KlfHprFeTdQFD7hibHNikqWVbjhH3aPmg1H90bZe+f+LfH98fHtsq4VO5JnvezaPjr+vw0TOQK9vYwE8/fTo5/Fmefu7N56tULLzyM37crX2dwjyio2Y/DPeb3+jUn8LHfVSfj0eloh9//lBvhP746JTQYYn2z8fH73UZ6b/Qfl6xBFITiXPjeG9NRubnqsfRuSfo2oRUmjjbwrClcwO6lGGrgOyOvBCBBKG3j/E0JWAZgm+u3NM7I+ugwgUxd4CjrEtLzdg/UZOmko+6g5lazPXMyftDt/3UKtxcruSCml/ZoXVimZM0kLeTRAmoAZwZA1edb03v5wF0nMyunUqQ5JbVap4bX8teAeBrHKNv3irdNXR0+0SqRr7zGJRrzuomFarXbfTqGZ4uY08xqYVuVal73tpUJq7Raz21PVte5JAdKZt+IE/gT4b9UytlBzW7tfAX9Btla4GrXKPv3z0qw/plPhTZ7vr3Y8L6+bhaGSM1GPa38EmZsp2Gm6265yTcJAVaoHTD42MDv8KNccssiPD26C49f/rbJ0ylNsA1kW7RJMreqRTbfpYY1dPHfdWqbYM+VNnmZkL497Rjnqa19QQxBNia+OM2z2DlVcFpiUAKU71oWXG4wvA2vHI8UtrcPvqIB5nbf1pCSU7rycNI75U7crYuwizZ2e//eUmDACyA8J8RHgjo/6fYAPhaYNCWXhHQzZBhYJ18/rkiQ/6SwPjTeUBbcoG0NdRmL7Uuxz1gN9Fqi/29J9w9xmACOtpGYcNQaV9nLgKfR8BL48WDXERr3zAZwijaxkclRL7fodHGX2OjkmU0J6nFdV7Y7lpqSVmuLUMJCY4X0TmuifCVH3KxwEUV/sDXCBbhSnj4j4MWxjmnmt+7d5CFC5B288TDDYDK9gF3S+8LONgoCMCuheFrcTHGwypi3Lzzg3wahio0Qzpj47zI3A7NeUxTTapbDNQNYN7/B7EGw/c=", "app.py": "eNrNO9ty3Lhy7/oKhK5TIrUUNaOLL7OZpGRb9vqUvVYk7dapzKq4GBIzQ4skuACpi1V6TJ5Slao8ppKq/EXynE85P5BfSDcAXoejGa99UpHLMxwQaACNvnfDsqx3ac7EjH4mN2xK7LeChhF3SMhiwmRewI8R/EhzwVVbynM25fyKcBLwhJOMzqOUbmXFNI4CCm0p/s8FlSylhBJ4umYShpIiJXmRAgh4fhXzIpzFVDDPsqytKMm4yMk85tPy+ZPkafnMZfkkaBrypPrFyifJAsHyqpssppngAZNVS74QDDaWzquGKGFb1cRq04RKeNramgmeEI+Yd7CbWTQ3jSyF3bLy1Zzlvm7Z2jr+8d2HczImE+sVTQAjwnKJ9YoLwdTTOY2LkJaPuX56SaNYP52xjEtuXW6dnp2cn1wgoPstAn9PAOHJVLARsTMquXQJD3LBmEsCCjh2Sc5u80JQR/W2zmgWhdyC3vtHLtk/euqSwwH8wfPg8Lnj6l7HsADscwjtB88PXfJ8uc8HehslqtcRvDka7rtkeKS7HQ5ePIVuD1snfzr+cPr+5Nx//e4MVmzt4ckDqez9UKR3BU0PXu/u71Ep4WT22C1Nspj5AHTOpLW1tRWyGfFnUcykHTojNSsXIRMKiQkHYuM+TaF/yL15PEU0mVYRzb3Z9LZu8aYxS8NGD42V6HM1VoGv/spuQOA0jwLu8emnxuiqOcnjRjOcumAy4ulB6El80wGaRewzleVaqQiiOKblT1izH6Uz7iFhw0HjiJnEvXLpZTRfeJ94lNqhS2YOmXFBZiRKDUKiGSk7sdtI5tJeHuNUIL8DmLMahARKZaGNvOXhx9JYa8dyHKe5F5hvhoyOwwEg8Jz6NQPyD+VNlC9s2/JiPrfcclpgvkJg5/JcZZEkVNzZuGVzuBQ2qzAAbGNb9FMhcyAEl9w/6Mlj4COFkGopM2tn574asq0oftsl23+77YzcB80BOzvkv/+z5AJyT3VX+L3tPJDsFl9aDYBM/lawmOWcNADD2WBvu9G0KBhwG0w2gBf6h0sy/CLA+i2QjVGZGbS9C/AcnLvxUrJ5AdJL+jnPaWz28UBkg45m1hsuEjoi99tJEefRNRw23cbzqME0XjiExZKR7SGcds63H1rLsvumniF4Nata3YVB27vjUXtL2zLqTGsw7Ef1tClO2TuN6VxP1AR+BlTExKh1AKqpRJxBiKas5iIsJQ4C4EAgHENVFeV4NMsAjG0d151gV+Q7guynqR0hTVpQLg3lt6aJeegr4lo9yfuPr6FlzgRfmqQlFQCXV4ik64pgHUvx5pVLrpG/9JLqGS+9KGeJtJ16YdY1EyCOmPRllPpIY75gGfQGVFsljBULnVmnSJMjMhwMyB9Q9caUJBQEE5HRvGCEopovecJqS7QuMU22H1nJ9uUDKV+TqtWxGsxdHxFoNLKzA1TMhbfIkxi4GI2Lv393SnAgAgIiI1rwEqMFCEuxMQWDYg4/BVEtx2eemQPRelPiwxAMToFCZnLprMTQn//1P/7nv/6Z3N88GEBGmlm/pL+k5lDVEMdItzlLmaA5s8EoSHOXTGkAxxmzGTyDLFnAl1wwVn75SorDK5bwa+ZP59Ces6zW5W2szyhgUCl26McYSGi0enz9OC+ikKYBUxO5ZMH0dAJESg6N11HIeAeeaoNzAphS6WAEDvvLKWjN0oCAB7CY5tBLjufCOzXPQIajkgzVZkkkyY/AOUopqN2VLTV6BY1APACUEyG4sK3zYqroLGEpiE+gPJRWSIoz4PucEZuDbUjJgn+i2Kpkm3TKU4WZq/3Xc+Av0BXaHvTwK0pzG22YnZ2DIdklQz0azDPoVhtqttO7nVGNriegMYHm2qup5gXqzFFJATRPZnGU+woJ9vJ51xpVzzTWYzVdqiazwVzcNXDHSuiiSH83ffWzcYfqSkJDxAEROCW19ZPYCpBtwtuU1lYAqygwpsk0BDpxSTKqWm34GTIZjBMjGtltwLKcnKgvEOdovzfIsDT0BWwTMVi9qFq8TMDmfYBjO6uoF1QySEvwge5ZKR4U4TE5scJIGEsOJU7Zis+mOYO2ci8j0mPobWTZaXWLZLpl+BnAVj9rW1F6MYCARVX2dEnqYK4i7SnTzdJnZDmKha15JPjB04FV2XujrY7wgMmWF9WUk3Zm99nrDlivpH7VNcmdNhU81hFfNs1p3dIwtzuwYNGqh96p2aGXZIc4cqIO6XOUWZeXoLcrD6QNom3AaiZB9nBKHTClebCw1WDDKIYF1olRZUnjsDUC85qKCAgaZaW0HhVmsBdYA9jjaDxfoq6rCCNyNW2wtEiMysKpG+RRcVdE9kDEpKaDYTU0SJWtDJgaPuzd1x1KZtBCMV/A3DNlRYFlANQG8gP5VebCkO/MS2nCGjx41yY00RB7CE8hdVwjdqw+W26K2nipxoU503YPQEpDz//bP5D7kpSnKkCRMDUZ2Kl//sd/6XkpJtsAdfuytd01cmd53n//p5XzNuWKYSd9nC3jg89LslsAh3MgS3OCn1FbPO7h6fiF9/GnC3TT0d3zEFGOgwoD7CzJxheiYO0lyMnoaHBp5gTy4IUAXVHO2jo8I2YzWeRRvCR8uQgWVeN1glpQdfSuI5EXNPYTUFq4n6oTUrEa5gVFSD147yPFIx/aSmA2XkbSp9cYRAHtYhtCQyNg0DisPo5pCi+w0A2KkOJFNGU28zLnAfF/dvwBDug68QoJ5saeMi4GI284e4Afy8YygIK+yr2rOw+g89uX6AIkXsYAjWn+8AflFL09/YkUkoZ01A/Lnk+Gl2DKzCeDS6c7+7162Z2l5WrVoOAEwWtEexwt5oAGC7B4FDMzT/2q6LuXtm9HXawBX9tgMpSysIji0C+ikj5WyKlTw+KZ+hVIpF2M/uEvT4fgdjGCRGGIIPcJvd29icJ8Ac7L4WAAvvxfaaqigEEdHANDnNyjJp8LDp7nSFn2VOwqYIBo+8UgZHP3ydH0aDobuE+eBYfhbOZ8TwIw8cAFvVmAs/U9SMo035XRZ0DK8Hl2q4GXK0OlibL5ZcyDK2nnUR6zsXV8Sj6guiIHrwmKSHAx8gVLGEp99SC9cz7LbbAwUI34iwIGRWClznlXW5k/wMcY/juI8RCYosY5gPxAxVXIb0D4Wk9Id+5fUiWkUYIZhwleNLSou9bDm1kBT0BkRMBHKoT7IbqlCQxsuMqg2ZTCJXdAEOoYAkUfK0G+jJVXDzT65uWf4PPt+5fw+fHlH+Hz/OI9fCqR7iGj/brEg6fOw68NmVuewgWd2taJjks3vfNmnzN+Y3deNV+/4nGRpLYMKBzkYU/Hps0O/d9hxNIGGcPisfWmdFmm4P1TEEURdfDs7zI43yxaCgqu+lNhUB+Pa2ydvX15bJWG9vhgMHB6YZTrPw4CMPgB97b1AY5DOygjOOOMxmBo7hGQMkpHw2Pbi1nl3y/9WXbCPqFdieEiPHADOwMzCOQV7piDchuDaSzZCgxucCLNP+TiHnyf6JnbKH4Ee8OnK7BXaWbwonrmOVU4I9Hn37xvN5fy1VZPBszxjSbTjvjyRD+0j99GN1tNoZI0oElDjEdtSLJK8zfWuvHC/DKsD8t7LXimBNmjc060g+wql1d5vwqVGGPTL7QXrN5gn3VbMKOwfw2jAo5N6kcXtnX5ONxrGqNYX7HWx8lQH9BHwExqonKKV214BiIsIiZCTJ8B4liwoI7Vj2N2u9b+ayZplPWXpXMM+E9Gw/3LXqBgYzVVfvcP/RSdzJE2xqiiNCtyOZ4oNID7UXLuJwZ9Kt9lA2n2isZRSMNNhQsmY0qqP0PjwUb31zYZNDBvzfGYTBZu63RiXQsKBuXUuiR/MyYHzzcnfWNd6uRZH7D9ge5SJuGcL+CrEmmngunEiOV8C6GqAj4aQ+eAWybsIVjHQ0ziwerVW6Bx1W08rE7uFBON1uOiTceQ2qBf7Lvk6aGGrd9XwA/2vwQZTYycgfMRF8rUwIihUki2hu58GySpMFgHSSbPqb9hO6pPtRvMg37pfsx2XmGQnySKJNfhOFfc3RSaE2s42D9EeYWpWvzGXCyuS9M6muWwWhgIjvDvXGGZE7Kz22+EYBOrhc4/FskU8bt/cFhR2zlLMLBjYeSEBRHmV8fr9GoZD9ZQXy1YcDXlt7aSGF3ABMw8baitQXcZ92yTAqa9XXLkDczZD7yjaoY30EWJLBDUJqz9bTBWRXI7G0RHvZr974oop2ijpSFfT0kYd30MWnnsaBe8O95caJ+UrkXDWQCLUZn2m0pyUJhNMT6xFkVCUw4gkMSDgoaiyFiI8Kw0SudFytep5mUt3YRZGZdm7d/m1LRB1CKfgQcyEQhm6D2v6GdQExDoEsQ5CP2Y2Imz5hR1rL0Ff3f4HAU6fpQE+mLwe0Xt20gAPcmIJPBAFnCUwDxYyUPseSuZ95V4UrTRkWyqfKZS2XUJTYmoirTQ79XE5ayfxBelZK9F6CEi69m+0oMKeYeHg0qAok4vB34Bia1VXJut2aRMWid8aMp4UF/r15UaqmWosZxQQ+pU+iMTVemYx6TBz+qIMHJOMDnQD22apxrGyyLPURa8VXlRgeEHLUbId6TmMRfMI5b4UTi2wAGBnxhxGVvxvAd8r6/+dAVdYbVOYbBWBUp0EB0dHpVtvCNZEUtKdnbMInd2vDVC7oJOpb2Bc6tCEsc68WGtIf1EZUjMUjFWc/Da7gojQFqzQqFytI72v5yzSQDKT/gq3DUGaTR4BoLCGzzHT1RtjrPZ9ox2WL+9ymbp7O6iDkd93Y6+0YaOdTpp/YaCmN717ugcs8TKHv3/cUSnKjq3fkNlBnt5Rz+lVMf4VImTKnr6yq39zr1tqEuuG8ykBJbd1RYbRNU0IBByPXDelvJvPRhe5DqTqMG8gccSymsG4kvMUY1ifYsJvCg2F8EiuuaSSOD8nH9RKAYnA8QWaT62VEkYqAYQaK3xGJunWRbf+dpXtjHx1INQrAlwMduOxovxnyfYdzk+UMb/yxHoceSdafVkXrCg6ZzZzQW4pPye9NcigPjoZPBAyXhBHAVXdll1swpHk79INc5fri7nSyt0VpojE61VXC1+XSO03JLXXcMorqZzt6ZV16hOvZ+V8IFig0KAYRHc+XGURDmaSTdcXDGhjJHm+1K7N0iiJaXec4yeN3PbTldktbT4qSrjpiQpgJzMCIJ1hoBu6ZEPmOVBC7VMNijNiWrffSTYbYGzhsm3O2pKuYgph0NorGsXTJcYu5f3XM2SGCGV44mlwqxWHQ5708jjbyjnpv7vc4x6PJ6tTa3Wx1yiqf81lvsSVpfMR3PWgsRAJJarKyAQxyaJtgRBlzw06aXbA0h9/bmZlYJwln2LNOJHFXyA4JqWvKOOxy2xcqlewXyuXtgjzsOX8MsPKt8f0bjLJYtl/B0HmFEHG6tr/i++CA9ndbKY4tWQiKMXZWNRPHAexmecJfg1mkyBgqsKlFw99arNnTHAg4o1dmpBlvf2M/B2IVXV4NnxB8wmnv7UWYWod7mCHkRjnVVRQ7lSoVe6ShItnaa1s/OKp5J94nJnRxVrLHfZJacgI3mK+bmMS4bZ9wvCyTEIzYKJTN2hyXT0RMVxSBxJeJ4WeHkmLj57K8Aq56bMyt3VGT9VPAscrMNROnkHj2VPSeM5TckU5NwqyL/2XI34FYSjuGYauiymkcCiXJKoJLEHW8CggcpZsL4coyUL2XRpvq8mMRc5fkVJDGKcmUp7ZSPpey/q5pDORq9a8Ums0smKPgG/SQQ6iuWCG2xUtzbALS1FO+bXPUBTwWywkSiQu79OySX01lcO69PDVqkMgtra2npCdr/iD84oBbbd8s9Pzn4+OVPXfx6qKzJglPiYcLfxw7CKZVmvAN+ourg6cLX9gOEpIzCueBeT9dMCzE+8fJUBxeVIYVNz66pRpyN5cMXyqmzsCmvGbCtKsZ4ClcsNmzaZFAOv5Vq9jGf2leYip1l1GGJxLVa7tauDe6u/FFDgTOCRDs92C1KWx2VU6vrcvLsqKy/SlGGusl4cFhD1Lyz3QNih9srLNSAqMkSFUJYsYsolCl/fkcNBAx9KqmkcevrLVnUccoNtS28apaFtW+AY4T8MhPf4RYbcsj7kfDxXhYPLsLGaJkoL1iRYXL8hrZgWabDw9TGr7Y2fPX9abkxtdLxEfxUDwbu68KdmKw3UlgwkhvDRixg3tmaa1Vwan3r2sQ4/yQUVTMeLl/VnhvVqae7ru3w+1uNUw/iNzxAHpqHMTQ/7EjVYz3vDQh9zpGCqdQvkmglT44yUVDUpWeISNq84f1kSKJ4NqvuNYVUUdctad+QaXaxmbWinHhiGNQitvtioCiVBQs2ZSovv/qY+P2KU7Xa1D0OsRZ5ncrS3NweSLaYovBsraS5qT4AdSEHh78XAETLfQzUYcxruPWJYN8bvAqKK212ahE8PlZeAgcZGqaHymiV4iaALcJcuGfBnR0ct4QrNbVIFOYT3ICUIZjCcYLMFENTYopmlGRPpd9iutS27N0peTQumzvW9US/nVyz1CxFLOmP2C+f/kgGaNE9okS/GNu7KrdbZk8nDbn4CZABORm8x2ua88lXsofWisid1NHd/MEimVpdrUH20WAYw3axhbwlGzShtFmqWKwd4bjUfnGKKyZ4oCqoFvrW7m/JdQBMvspAqdwKaYNpNfKGZ4hJgkuH+M3Vww9E9ntXDJikBmYdgSI6bK3x3eoKuNvinotl+fvEasKr9+w5jIN3hDWUPPxrbv1kAqptvyC70/WvydNCpP8ZbymOFLU8vyEM6iNslsCogqK4teJJhMMoWlXyY0N3Pg90Xu5ff/eLB8dSn8QvKDPQWcAq8Ctkxw4HtkmVNpA888bBQM7N7gmhglNDGFY3yurZ3oZ7snAqQdOZiyIhM1B0k1M8+6ufGPoEkQwpklmqM6psPjT3XRGlo5dLgaZMrJeq+iG3Vt9dBwpAwkhlPsXRyhNK3MjNwy0t3owyEnyReXMLbbCyNaQCmtbo/r/wbfQNf364X2nLnnteMSigx0zWW+iXSklRSwkZLgn5Zs1p09JL+lyvTmhzUwtSKUPT2HNGS3DB6AXpr0V8vfOt/AUuzAZY="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0/*"]
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv/*"], max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()